In [ ]:
# install dependencies
%pip install anthropic python-dotenv

In [4]:
# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

In [24]:
# Import the Anthropic client and set up the model
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [54]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system_message=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000, 
        "messages": messages,
        "temperature": temperature,
    }

    if system_message:
        params["system"] = system_message

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)

    return message.content[0].text

In [6]:
messages = [] # consider it as a conversation history

add_user_message(messages, "Define quantum computing in one sentence.")

answer = chat(messages)
 
add_assistant_message(messages, answer) 

add_user_message(messages, "Write another sentence")

answer = chat(messages)
answer


'Unlike classical computers that use bits representing either 0 or 1, quantum computers use quantum bits, or qubits, which can exist in multiple states simultaneously, enabling them to perform many calculations at once.'

# Simple Chat App


In [ ]:

messages = []

while True:
    user_input = input("You: ")
    print("You: ", user_input)
    add_user_message(messages, user_input)
    
    answer = chat(messages)
    add_assistant_message(messages, answer)
    
    print("-" * 30)
    print(f"Assistant: {answer}")
    print("-" * 30)

You:  what is 1 + 1
------------------------------
Assistant: 1 + 1 = **2**
------------------------------
You:  add 2 
------------------------------
Assistant: 2 + 2 = **4**
------------------------------


# System Prompts


In [ ]:
# math teacher
system = """
    You are a patient math tutor.
    Do not directly anser a student's questions.
    Guide them to a solttion step by step.
    """

messages = []
add_user_message(messages, "How do I solve the equation 2x + 3 = 7?")

answer = chat(messages, system_message=system)
answer


"Great question! Let's work through this together step by step.\n\nThe goal is to **get x by itself** on one side of the equation.\n\nLet's start with the first step:\n\n**Look at the equation: 2x + 3 = 7**\n\nRight now, there's a **+ 3** on the same side as x. What do you think we should do to get rid of that + 3?\n\n*Hint: Think about what operation is the opposite of adding 3.* 😊"

In [18]:
system = """
        You are a python engineer who write very concise code.
    """

messages = []
add_user_message(messages, "Write a Python fuction that checks strings for dublicated characters.")

answer = chat(messages)
print("Before system message: ", answer)

print("-" * 300)

answer = chat(messages, system_message=system)
print("After system message: ", answer)



Before system message:  ## Function to Check for Duplicate Characters in a String

Here's a Python function that checks strings for duplicated characters:

```python
def check_duplicates(string: str) -> dict:
    """
    Checks a string for duplicated characters.

    Args:
        string (str): The input string to check.

    Returns:
        dict: A dictionary containing:
            - 'has_duplicates' (bool): True if duplicates exist, False otherwise.
            - 'duplicates' (dict): Characters and their counts (only duplicated ones).
            - 'unique_chars' (set): Characters that appear only once.
    """
    char_count = {}

    # Count occurrences of each character
    for char in string:
        char_count[char] = char_count.get(char, 0) + 1

    # Separate duplicates from unique characters
    duplicates = {char: count for char, count in char_count.items() if count > 1}
    unique_chars = {char for char, count in char_count.items() if count == 1}

    return {
        "h

# Temperature

In [ ]:
# Temperature: A higher temperature (e.g., 0.8) will make the output more random and creative 
# , while a lower temperature (e.g., 0.2) will make it more focused and deterministic (real).

messages = []

add_user_message(messages, "Genarate one sentence movie idea.")

answer = chat(messages, temperature=0.0)
print("With Tempreture 0: ", answer)
print("-" * 30)

answer = chat(messages, temperature=1.0) # increase the chances of getting different output
print("With Tempreture 1: ", answer)


With Tempreture 0:  Here's a one sentence movie idea:

**A retired safecracker with early-stage Alzheimer's must pull off one final heist to fund his own care — before he forgets the combination.**
------------------------------
With Tempreture 1:  Here's a one sentence movie idea:

**A burned-out lighthouse keeper discovers that the mysterious light signals coming from the ocean each night are actually messages from his future self, warning him about a catastrophic event he must prevent.**


# Response Streaming 

In [ ]:

messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01SLZLdUg5Q1TwtZ3x43hEhF', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a 1 sentence description of a fake database:\n\n**"Nova', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='Base is a fictional cloud-based database m

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

stream.get_final_message() # to get the final message after the stream is done


Here is a one sentence description of a fake database:

**"NovaBase is a fictional cloud-based relational database management system developed by the made-up company Synthetix Technologies, designed to store and organize imaginary customer records, fake transaction histories, and non-existent product inventories for hypothetical businesses."**

ParsedMessage(id='msg_01CrcdMGRMWnGvdqD6BPoBDn', container=None, content=[ParsedTextBlock(citations=None, text='Here is a one sentence description of a fake database:\n\n**"NovaBase is a fictional cloud-based relational database management system developed by the made-up company Synthetix Technologies, designed to store and organize imaginary customer records, fake transaction histories, and non-existent product inventories for hypothetical businesses."**', type='text', parsed_output=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=73, server_tool_use=None, service_tier='standard'))

# Structured data


In [55]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages, system_message="Respond with raw JSON only. No markdown, no explanation.")
print(text)

```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}
```
